# Experiment 8.1.1 — Two-layer multi-threshold factorial

Analysis-only notebook. It compares the 2×2 L1/L2 Binary-vs-MT3 factorial under the locked Exp7.3 A2 training contract. No training is performed here.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_8_1_1_two_layer_mt_factorial' / 'two_layer_mt_factorial_v1'

manifest = json.loads((ART / 'manifest.json').read_text())
runs = pd.read_csv(ART / 'method_runs.csv')
summary = pd.read_csv(ART / 'method_summary.csv')
contrasts = pd.read_csv(ART / 'contrast_summary.csv')
effects = pd.read_csv(ART / 'factorial_effect_summary.csv')
activity = pd.read_csv(ART / 'activity_summary.csv')
manifest


## Method-level local representation

The primary endpoint is L2 communicated Fixed250 probe BA. Native Linear/WCCE test BA remains secondary.


In [ ]:
cols = [
    'method',
    'linear_test_ba_mean',
    'l1_pre_reset_fixed250_ba_mean',
    'l1_communication_fixed250_ba_mean',
    'l2_pre_reset_fixed250_ba_mean',
    'l2_communication_fixed250_ba_mean',
    'l1_threshold_delta_fixed250_mean',
    'l1_to_l2_transform_delta_fixed250_mean',
    'l2_threshold_delta_fixed250_mean',
]
summary[[c for c in cols if c in summary.columns]]


In [ ]:
order = manifest['method_order']
plot_df = runs.groupby('method', sort=False)['l2_communication_fixed250_ba'].agg(['mean', 'std']).reindex(order)
ax = plot_df['mean'].plot(kind='bar', yerr=plot_df['std'], capsize=4, figsize=(7, 4))
ax.set_ylabel('L2 communication Fixed250 BA')
ax.set_xlabel('Method')
ax.set_title('Exp8.1.1 local representation after L2 thresholding')
plt.tight_layout()
plt.show()


## Information-flow view

Compare L1 pre-reset → L1 communication → L2 pre-reset → L2 communication for each factorial cell.


In [ ]:
flow_cols = [
    'l1_pre_reset_fixed250_ba',
    'l1_communication_fixed250_ba',
    'l2_pre_reset_fixed250_ba',
    'l2_communication_fixed250_ba',
]
flow = runs.groupby('method', sort=False)[flow_cols].mean().reindex(manifest['method_order'])
flow


In [ ]:
ax = flow.plot(kind='bar', figsize=(10, 4))
ax.set_ylabel('Fixed250 probe BA')
ax.set_xlabel('Method')
ax.set_title('Layer-wise local information flow')
plt.tight_layout()
plt.show()


## Seed-paired contrasts

`MM - MB` is the direct test of whether changing only L2 from Binary to MT3 preserves an MT-L1 representation. `BM - BB` tests the same L2 change when L1 remains Binary.


In [ ]:
contrast_cols = [
    c for c in contrasts.columns
    if c in {'contrast', 'method_a', 'method_b'}
    or c.startswith('delta_l2_communication_fixed250_ba')
    or c.startswith('delta_l2_pre_reset_fixed250_ba')
    or c.startswith('delta_linear_test_ba')
]
contrasts[contrast_cols]


## 2×2 factorial effects

The interaction is `MM - MB - BM + BB`. A positive interaction on L2 communicated Fixed250 BA means L2 MT is especially useful when L1 is already MT.


In [ ]:
effects[effects['metric'].isin([
    'l2_communication_fixed250_ba',
    'l2_pre_reset_fixed250_ba',
    'linear_test_ba',
])].reset_index(drop=True)


## Threshold × tau activity

Inspect L2 MT activity for saturation of the 0.5× threshold bank before interpreting a negative MT result.


In [ ]:
activity_cols = [
    'method', 'layer', 'threshold_multiplier', 'shift',
    'mean_value_per_neuron_step_mean', 'fraction_nonzero_mean',
]
activity[[c for c in activity_cols if c in activity.columns]].sort_values(
    ['method', 'layer', 'threshold_multiplier', 'shift']
).reset_index(drop=True)
